In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mosapabdelghany/adult-income-prediction-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\dsapu\.cache\kagglehub\datasets\mosapabdelghany\adult-income-prediction-dataset\versions\1


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, precision_recall_curve, auc, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from skopt.space import Real, Integer, Categorical
from scipy.stats import randint, uniform
from skopt import BayesSearchCV
from sklearn.pipeline import Pipeline
from category_encoders import BinaryEncoder, TargetEncoder, WOEEncoder
from sklearn.impute import SimpleImputer

import optuna

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, ElasticNet


In [3]:
df = pd.read_csv('adult.csv')
df

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [4]:
df.isnull().sum()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [5]:
df.income.value_counts()

income
<=50K    24720
>50K      7841
Name: count, dtype: int64

In [6]:
df.income.map({'<=50K': 1, '>50K':0})

0        1
1        1
2        1
3        1
4        1
        ..
32556    1
32557    1
32558    0
32559    1
32560    1
Name: income, Length: 32561, dtype: int64

In [7]:
X = df.drop(columns='income', axis=1)
y = df.income

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((26048, 14), (6513, 14), (26048,), (6513,))

In [8]:
cat = [col for col in X_train.columns if X_train[col].dtype == 'object' or X_train[col].nunique() < 30]
num = [col for col in X_train.columns if col not in cat]
cat, num

(['workclass',
  'education',
  'education.num',
  'marital.status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native.country'],
 ['age', 'fnlwgt', 'capital.gain', 'capital.loss', 'hours.per.week'])

In [9]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('std_scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num),
    ('cat', cat_pipe, cat)
], remainder='passthrough')


In [10]:
def objective(trial):
    params = {
        'C': trial.suggest_float('C', 1e-4, 1.0, log=True),
        'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 1.0)
    }
    model = LogisticRegression(**params, random_state=42, solver='saga', max_iter=10000000, class_weight='balanced')

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    score = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, n_jobs=-1, show_progress_bar=True)

[I 2025-09-16 20:18:52,397] A new study created in memory with name: no-name-69a09271-f764-43dd-ae52-978453b9aa1e


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-09-16 20:19:04,460] Trial 4 finished with value: 0.8976425345703929 and parameters: {'C': 0.01647408340323824, 'l1_ratio': 0.5267839275513947}. Best is trial 4 with value: 0.8976425345703929.
[I 2025-09-16 20:19:04,679] Trial 14 finished with value: 0.8867480386163475 and parameters: {'C': 0.0022513514860221955, 'l1_ratio': 0.7889824703952553}. Best is trial 4 with value: 0.8976425345703929.
[I 2025-09-16 20:19:04,888] Trial 10 finished with value: 0.8692388411419841 and parameters: {'C': 0.00023697780312701303, 'l1_ratio': 0.7845311212293121}. Best is trial 4 with value: 0.8976425345703929.
[I 2025-09-16 20:19:05,244] Trial 11 finished with value: 0.875247869058048 and parameters: {'C': 0.0005407987669649241, 'l1_ratio': 0.6834018544080095}. Best is trial 4 with value: 0.8976425345703929.
[I 2025-09-16 20:19:05,804] Trial 0 finished with value: 0.8960524920523287 and parameters: {'C': 0.010154945025547802, 'l1_ratio': 0.7421319415270836}. Best is trial 4 with value: 0.89764253

In [11]:
best_params = study.best_params

In [13]:
model = LogisticRegression(**best_params, random_state=42, solver='saga', max_iter=10000000, class_weight='balanced')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])
    
score_train = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc').mean()
score_test = cross_val_score(pipeline, X_test, y_test, cv=5, scoring='roc_auc').mean()
score_train, score_test

(0.8998267665972552, 0.896055861541526)